# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

### Dataset Source

This dataset is described by a Croissant schema, accessible at the following URL:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

**Note:** All RecordSets, Fields, and Columns will be referenced by their `@id` for reproducibility and reliability.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading

Load dataset metadata and explore general dataset information.

In [ ]:
# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print the dataset metadata as a summary
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}")

## 2. Data Overview

Explore available record sets, their IDs, and field IDs defined in the dataset. All entities are referenced using their `@id`.

In [ ]:
# List record sets and their field @id(s)
if hasattr(dataset.metadata, "recordSet") and dataset.metadata.recordSet:
    print("Record Sets and Fields:")
    record_sets = dataset.metadata.recordSet
else:
    # fall back: auto-detect from available distributions in most Croissant schema
    record_sets = [record_set['@id'] for record_set in dataset.record_sets]

if not isinstance(record_sets, list):
    record_sets = [record_sets]

for rs_id in record_sets:
    print(f"Record Set @id: {rs_id}")
    rs = dataset.get_record_set(rs_id)
    if hasattr(rs, 'field'):
        print("  Fields:")
        for field in rs.field:
            print(f"    Field @id: {field.get('@id', None)} | name: {field.get('name', None)} | dataType: {field.get('dataType', None)}")
    else:
        print("  No fields found in this record set.")

## 3. Data Extraction

Load the records from a chosen record set into a pandas DataFrame for analysis. Use the record set and field `@id` values identified above.

In [ ]:
# Identify main record set to extract (replace with primary @id from above)
# If record_sets is empty, try to guess primary record set
if not record_sets or (len(record_sets)==1 and not record_sets[0]):
    # fallback: auto-detect
    all_record_sets = [rs['@id'] for rs in dataset.record_sets]
else:
    all_record_sets = record_sets

dataframes = {}
for rs_id in all_record_sets:
    print(f"Extracting rows from record set {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} records for {rs_id}.")
        print(f"Example columns: {list(dataframes[rs_id].columns)}")
        display(dataframes[rs_id].head(2))

# We'll select the first available record set for further analysis
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
else:
    raise ValueError("No data extracted. Check the Croissant schema and record set definitions.")
print(f"Using main record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Demonstrate exploratory steps:
- Select a numeric field (by `@id`/column name) for filtering
- Filter records (e.g., protein abundance above threshold)
- Normalize a numeric column and optionally group by a categorical variable

In [ ]:
# Guess/choose a numeric field to analyze
print("All available columns in DataFrame:")
print(df.columns.tolist())

# Let's heuristically pick common mass spectrometry field names (substitute with your schema's @id if different)
candidate_numeric_fields = [
    col for col in df.columns if any(k in col.lower() for k in ["abundance", "coverage", "peptide", "molecular", "mw", "intensity"])
]

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    numeric_field = df.select_dtypes(include=np.number).columns[0] if not df.select_dtypes(include=np.number).empty else df.columns[0]
print(f"Selected numeric field for EDA: {numeric_field}")

# Ensure numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

# Pick a group field if available
group_fields = [c for c in df.columns if c not in [numeric_field] and df[c].nunique() > 1 and df[c].dtype==object]
if group_fields:
    group_field = group_fields[0]
    print(f"Grouping by {group_field}:")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    display(grouped_df.head())

## 5. Visualization

Visualize numeric field distributions and (optionally) grouped means or relationships.

In [ ]:
# Distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group field exists, plot group means
if 'group_field' in locals():
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion

- Successfully loaded and explored the FAIR^2 mass spectrometry dataset using the Croissant schema and the `mlcroissant` library.
- Inspected record sets, fields, and extracted data using only entity `@id` references, ensuring reproducibility.
- Demonstrated basic exploratory processing, normalization, grouping, and visualization.

You can now proceed to domain-specific analyses such as protein annotation, comparing abundances, or exporting data for downstream bioinformatics pipelines!